<a href="https://colab.research.google.com/github/mansha5/dyslexic_handwriting_analysis_ai_tool/blob/main/Model_dyslexia_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets sentencepiece accelerate jiwer tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.5 MB/s eta 0:00:00


In [ ]:
import random, math, os
from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments
import pandas as pd
import torch
from tqdm import tqdm
from jiwer import wer, cer
device = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", device)


DEVICE: cuda


In [ ]:
model_name = "t5-base"
num_synthetic_per_line = 3
target_dataset_size = 10000
max_length = 128
per_device_train_batch_size = 4
gradient_accumulation_steps = 4
num_train_epochs = 8
learning_rate = 3e-4
output_dir = "./ocr_corrector_t5_base"

In [ ]:
print("Loading Teklia/IAM-line (this may take a little)...")
ds = load_dataset("Teklia/IAM-line", split="train")
lines = [ex["text"].strip() for ex in ds if ex["text"] and len(ex["text"].split()) > 1]

# keep only reasonably short lines (avoid very long lines)
lines = [l for l in lines if len(l) < 120]
print(f"Available clean lines: {len(lines)}")

# if not enough lines, fall back to wikitext
if len(lines) < 200:
    print("Too few lines found in Teklia/IAM-line; falling back to wikitext-2-raw-v1")
    ds2 = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
    lines = [l["text"].strip() for l in ds2 if l["text"] and len(l["text"].split()) > 3]
    lines = [l for l in lines if len(l) < 120]
    print("Fallback lines:", len(lines))


Loading Teklia/IAM-line (this may take a little)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train.parquet:   0%|          | 0.00/167M [00:00<?, ?B/s]

data/validation.parquet:   0%|          | 0.00/24.7M [00:00<?, ?B/s]

data/test.parquet:   0%|          | 0.00/73.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6482 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/976 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2915 [00:00<?, ? examples/s]

Available clean lines: 6480


In [ ]:
print("\nSample lines from the dataset:")
for l in lines[:10]:   # first 10 lines
    print("-", l)


Sample lines from the dataset:
- put down a resolution on the subject
- and he is to be backed by Mr. Will
- nominating any more Labour life Peers
- M Ps tomorrow. Mr. Michael Foot has
- Griffiths, M P for Manchester Exchange .
- is to be made at a meeting of Labour
- A MOVE to stop Mr. Gaitskell from
- 0M P for Manchester Exchange .
- A MOVE to stop Mr. Gaitskell from nominating
- meeting of Labour 0M Ps tommorow . Mr. Michael


In [ ]:
import random

# --- CONFUSION MAPS ---

# OCR-style (visual confusions)
ocr_confusions = {
    "m": ["rn"], "rn": ["m"],
    "o": ["0", "°"], "0": ["o"],
    "i": ["l", "1"], "l": ["i", "1"],
    "u": ["v"], "v": ["u"],
    "c": ["e"], "e": ["c"],
    "s": ["5", "$"], "5": ["s"],
    "g": ["q", "9"], "q": ["g"],
    "t": ["f"], "h": ["b"],
    "a": ["@", "ä"],
}

# Dyslexia-style (cognitive/phonetic confusions)
dys_confusions = {
    "b": ["d", "p"], "d": ["b", "q"], "p": ["q", "b"], "q": ["p", "d"],
    "t": ["d"], "f": ["ph"], "ph": ["f"],
    "k": ["c"], "c": ["k", "s"], "s": ["c", "z"], "z": ["s"],
    "m": ["n"], "n": ["m"], "r": ["l"], "l": ["r"],
    "v": ["w"], "w": ["v"], "g": ["j"], "j": ["g"],
    "y": ["i"], "i": ["y"],
}

# --- HYBRID NOISE FUNCTION ---

def inject_noise(
    text,
    mode="hybrid",  # "ocr", "dyslexic", or "hybrid"
    prob_confuse=0.15,
    prob_delete=0.05,
    prob_insert=0.05,
    prob_swap=0.07,
    prob_space=0.06,
    prob_case=0.05
):
    """Inject OCR-like, Dyslexic-like, or hybrid noise into text."""

    if mode == "ocr":
        confusion_map = ocr_confusions
    elif mode == "dyslexic":
        confusion_map = dys_confusions
    else:  # hybrid mode
        confusion_map = {**ocr_confusions, **dys_confusions}

    noisy = ""
    for ch in text:
        # random deletion
        if random.random() < prob_delete and ch != " ":
            continue

        # confusion substitution
        if ch.lower() in confusion_map and random.random() < prob_confuse:
            sub = random.choice(confusion_map[ch.lower()])
            noisy += sub.upper() if ch.isupper() else sub
        else:
            noisy += ch

        # random insertion (like dyslexic doubling)
        if random.random() < prob_insert:
            noisy += ch

    # transpositions
    if random.random() < prob_swap and len(noisy) > 3:
        i = random.randint(0, len(noisy) - 2)
        noisy = noisy[:i] + noisy[i+1] + noisy[i] + noisy[i+2:]

    # random spacing errors
    if random.random() < prob_space:
        idx = random.randint(0, len(noisy) - 1)
        noisy = noisy[:idx] + " " + noisy[idx:]
    if random.random() < prob_space and " " in noisy:
        noisy = noisy.replace(" ", "", 1)

    # case irregularity
    if random.random() < prob_case and len(noisy) > 2:
        idx = random.randint(0, len(noisy) - 1)
        noisy = noisy[:idx] + noisy[idx].swapcase() + noisy[idx+1:]

    return noisy

# --- DEMO TEST ---
samples = [
    "This is a test sentence",
    "Machine learning for OCR is fun",
    "Dyslexic handwriting is hard to read",
    "The quick brown fox jumps over the lazy dog",
    "Please write your name on the paper"
]

for s in samples:
    print("Clean:", s)
    print("OCR Noise :", inject_noise(s, mode="ocr"))
    print("Dyslexic :", inject_noise(s, mode="dyslexic"))
    print("Hybrid :", inject_noise(s, mode="hybrid"))
    print("-" * 60)


Clean: This is a test sentence
OCR Noise : Tbis is a tesf sntenece
Dyslexic : Thyc iis a ttesd sentence
Hybrid : Thiz iz a testt sentence
------------------------------------------------------------
Clean: Machine learning for OCR is fun
OCR Noise : Machine learnin gfor OC ls fun
Dyslexic : Macchinne learninng phol OCRR  iz fn
Hybrid : achine reanimj for OCR  isf unn
------------------------------------------------------------
Clean: Dyslexic handwriting is hard to read
OCR Noise : DDyssllcxic bandwriting is hard t° read
Dyslexic : Dyslexic hamdwrityng  is hrd to rad
Hybrid : Dylexyc hamdwritimg is bad t lead
------------------------------------------------------------
Clean: The quick brown fox jumps over the lazy dog
OCR Noise : Thee qvuickk brown fox jums oveer tbe lazy d°g
Dyslexic : Thhe quick bbrowm  fox gunppc ower the laay dooj
Hybrid : The quuick brown f0x jumps ovee the l@zy do
------------------------------------------------------------
Clean: Please write your name on the p

In [ ]:
lines = samples
target_dataset_size = 2000

print("\nBuilding synthetic dataset...")
pairs = []
pbar = tqdm(total=target_dataset_size)
while len(pairs) < target_dataset_size:
    line = random.choice(lines)
    noisy = inject_noise(line, mode="hybrid")
    if len(noisy.strip()) == 0 or noisy.strip() == line.strip():
        continue
    pairs.append({"noisy": noisy, "clean": line})
    pbar.update(1)
pbar.close()
print("Total pairs:", len(pairs))

df = pd.DataFrame(pairs)
dataset = Dataset.from_pandas(df)
split = dataset.train_test_split(test_size=0.05, seed=42)
train_ds, eval_ds = split["train"], split["test"]

print("Train size:", len(train_ds), "Eval size:", len(eval_ds))



Building synthetic dataset...


100%|██████████| 2000/2000 [00:00<00:00, 35286.58it/s]

Total pairs: 2000
Train size: 1900 Eval size: 100


In [ ]:
model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

max_length = 64

def preprocess(batch):
    inputs = tokenizer(batch["noisy"], padding="max_length", truncation=True, max_length=max_length)
    labels = tokenizer(batch["clean"], padding="max_length", truncation=True, max_length=max_length)
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]
    ]
    inputs["labels"] = labels["input_ids"]
    return inputs

train_tokenized = train_ds.map(preprocess, batched=True, remove_columns=["noisy","clean"])
eval_tokenized = eval_ds.map(preprocess, batched=True, remove_columns=["noisy","clean"])
print("Tokenized columns:", train_tokenized.column_names)


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Map:   0%|          | 0/1900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenized columns: ['input_ids', 'attention_mask', 'labels']


In [ ]:
output_dir = "./text_correction_hybrid"
learning_rate = 3e-4
per_device_train_batch_size = 8
gradient_accumulation_steps = 2
num_train_epochs = 3

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    logging_steps=100,
    logging_dir="./logs",
    learning_rate=learning_rate,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    num_train_epochs=num_train_epochs,
    predict_with_generate=True,
    save_total_limit=3,
    fp16=True if device=="cuda" else False,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print("Starting training...")
trainer.train()
print("Training finished. Saving model...")
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)



Starting training...


/tmp/ipython-input-1788374340.py:26: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Step,Training Loss,Validation Loss


Training finished. Saving model...


('./text_correction_hybrid/tokenizer_config.json',
 './text_correction_hybrid/special_tokens_map.json',
 './text_correction_hybrid/spiece.model',
 './text_correction_hybrid/added_tokens.json',
 './text_correction_hybrid/tokenizer.json')

In [ ]:
print("\nRunning inference on eval set and computing WER/CER...")

def model_correct(text):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length).to(device)
    out = model.generate(**enc, max_length=max_length, num_beams=4)
    return tokenizer.decode(out[0], skip_special_tokens=True)

sample_eval = eval_ds.select(range(min(50, len(eval_ds))))
rows, wer_list, cer_list = [], [], []

for ex in tqdm(sample_eval):
    noisy, clean = ex["noisy"], ex["clean"]
    pred = model_correct(noisy)
    rows.append({"noisy": noisy, "clean": clean, "pred": pred})
    try:
        wer_list.append(wer(clean, pred))
        cer_list.append(cer(clean, pred))
    except Exception:
        pass

results_df = pd.DataFrame(rows)
print(f"\nAvg WER (sample): {sum(wer_list)/len(wer_list):.3f}  Avg CER (sample): {sum(cer_list)/len(cer_list):.3f}")
print(results_df.head(10))

results_df.to_csv("correction_results_sample.csv", index=False)
print("Saved correction_results_sample.csv and model at", output_dir)


Running inference on eval set and computing WER/CER...


100%|██████████| 50/50 [00:09<00:00,  5.38it/s]


Avg WER (sample): 0.000  Avg CER (sample): 0.000
                                            noisy  \
0              Please wite your name on th pacper   
1    Thc quick blrwn f0x junps  owcr thc lasy q°g   
2     Tbe puick  broown fox gumps ver the lazy oj   
3  The  quick plown f0x jvmps ovel  dbe razy ddog   
4                          his is a test ssentenc   
5              Machiine learning f°r OKR R is fvn   
6            Dyslexic handwr iitinj i bard to red   
7       he qvick br0wn fox jumq over thc razy dog   
8            Qlease wwrite your name on the qbeer   
9                 Masbne lcarning for OKCR ys fun   

                                         clean  \
0          Please write your name on the paper   
1  The quick brown fox jumps over the lazy dog   
2  The quick brown fox jumps over the lazy dog   
3  The quick brown fox jumps over the lazy dog   
4                      This is a test sentence   
5              Machine learning for OCR is fun   
6         Dyslex

In [ ]:
from google.colab import files
import shutil

# Step 1: Zip your model folder
shutil.make_archive('text_correction_hybrid', 'zip', './text_correction_hybrid')

# Step 2: Download the zip file
files.download('text_correction_hybrid.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>